In [1]:
!pip install -q \
agno==2.6.2 \
google-genai \
sqlalchemy \
pandas \
tabulate \
python-dotenv

In [2]:
import os
import sqlite3
import pandas as pd

from dotenv import load_dotenv

from agno.agent import Agent

from agno.models.google import Gemini

from agno.tools import tool

from agno.knowledge.knowledge import Knowledge

from agno.vectordb.lancedb import LanceDb

from agno.knowledge.embedder.sentence_transformer import (
    SentenceTransformerEmbedder
)

In [4]:
load_dotenv()

GOOGLE_API_KEY =""

if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY missing")

print("Environment Ready")

Environment Ready


In [12]:
conn = sqlite3.connect("../commercial.db")

In [13]:
pd.read_sql(

    "SELECT * FROM sales LIMIT 5",

    conn

)

,DATE,STORE_ID,SKU_ID,SALES_VALUE,QUANTITY,PROMOTION,SKU_NAME,BRAND,CATEGORY,PACK_SIZE,MRP,STORE_NAME,RETAILER,STATE,REGION,CHANNEL
0,2024-01-01,77,1001,740,37,1,Pringles Sour Cream,Pringles,Snacks,150,20,"Beck, Powers and Andrade",Reliance Fresh,Maharashtra,East,Modern Trade
1,2024-01-01,16,1006,480,48,1,Oats,Kellogg's,Breakfast,50,10,"Lewis, Garcia and Jones",Metro Cash & Carry,UP,South,E-Commerce
2,2024-01-01,15,1007,620,31,1,Granola,Kellogg's,Breakfast,50,20,Jacobson-Burnett,Big Bazaar,Delhi,South,General Trade
3,2024-01-01,41,1005,1700,34,1,Muesli,Kellogg's,Cereal,250,50,"Nelson, Cochran and Mitchell",Star Bazaar,Telangana,West,Modern Trade
4,2024-01-01,46,1006,280,28,0,Oats,Kellogg's,Breakfast,50,10,Howard LLC,More,UP,West,Modern Trade


In [15]:
embedder = SentenceTransformerEmbedder(
    id="BAAI/bge-small-en-v1.5"
)

vector_db = LanceDb(

    uri="../vectordb",

    table_name="commercial_knowledge",

    embedder=embedder

)

knowledge = Knowledge(

    name="Commercial Knowledge",

    vector_db=vector_db,

    max_results=5

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
docs = knowledge.search(

    query="What is Promotion ROI?"

)

len(docs)

INFO Found 5 documents

5

In [17]:
for doc in docs:

    print("="*80)

    print(doc.name)

    print()

    print(doc.content[:500])

Trade_Promotion_Guidelines

Trade Promotion Guidelines Trade Promotion Guidelines Discounts should normally remain between 10% and 20%. Promotions should run for a maximum of four weeks. Evaluate promotion ROI after campaign completion. Monitor cannibalization across nearby SKUs. Prefer feature displays during launch periods. Checkout displays generally improve conversion. Review incremental sales rather than gross sales. Measure uplift against historical baseline.
Promotion_SOP

Trade Promotion SOP Purpose This document describes the approval process for trade promotions. Promotion Types Temporary Price Reduction Bundle Offers BOGO Display Promotion Festival Promotion Approval Matrix Less than 5% discount Category Manager 5-15% Commercial Director More than 15% Finance Approval Required Promotion Evaluation Every promotion must be evaluated using Incremental Sales ROI Promotion Lift Cannibalization
Commercial_FAQs

Commercial FAQs Commercial FAQs What is Numeric Distribution? Numeric 

In [22]:
model = Gemini(
    id="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY
)

print("Gemini Model Loaded")

Gemini Model Loaded


In [26]:
from agno.tools import tool
from google import genai
import pandas as pd

client = genai.Client(api_key=GOOGLE_API_KEY)


@tool
def sales_analytics(question: str) -> str:
    """
    Answers questions about commercial sales data.

    Input:
        Natural language question

    Output:
        SQL results formatted as markdown.
    """

    prompt = f"""
You are an expert SQLite SQL generator.

Database Schema

{DATABASE_SCHEMA}

Rules

1. Return ONLY SQL.
2. SQLite syntax only.
3. Never explain.
4. Never use markdown.
5. Always use valid column names.

User Question

{question}
"""

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        sql = response.text.strip()

        # Remove markdown if Gemini returns it
        sql = sql.replace("```sql", "")
        sql = sql.replace("```", "")
        sql = sql.strip()

        print("=" * 80)
        print("Generated SQL")
        print("=" * 80)
        print(sql)
        print()

        df = pd.read_sql(sql, conn)

        if df.empty:
            return "No records found."

        return df.to_markdown(index=False)

    except Exception as e:

        return f"SQL Tool Error:\n{str(e)}"

In [29]:
agent = Agent(

    name="Commercial Intelligence Copilot",

    model=model,

    knowledge=knowledge,

    search_knowledge=True,

    add_search_knowledge_instructions=True,

    markdown=True,

    tools=[
        sales_analytics
    ],

    instructions=f"""
You are an AI Commercial Intelligence Assistant.

You have two capabilities.

------------------------------------------
Sales Analytics
------------------------------------------

Whenever users ask about

• Sales
• Revenue
• Retailers
• SKU
• Products
• Promotion
• Ranking
• Quantity
• Region
• State

use the sales_analytics tool.

------------------------------------------
Knowledge Base
------------------------------------------

Whenever users ask about

• Ingredients
• Promotion Guidelines
• Retailer Matching
• Commercial Policies
• FAQs
• Distribution
• RGM
• Assortment

search your Knowledge Base.

------------------------------------------

If a question requires both,
first use Sales Analytics,
then search the Knowledge Base.

Examples

Q:
Which SKU generated the highest sales and what allergens does it contain?

↓

Call sales_analytics

↓

Take SKU

↓

Search Knowledge

↓

Combine into one answer.

Always include

1. Answer

2. Business Insight

3. Source

Database Schema

{DATABASE_SCHEMA}

"""
)

In [30]:
response = agent.run(

"""
Which retailer generated the highest sales?
"""

)

print(response.content)

Generated SQL
SELECT RETAILER FROM sales GROUP BY RETAILER ORDER BY SUM(SALES_VALUE) DESC LIMIT 1

Reliance Fresh generated the highest sales.

**Business Insight:** Identifying the top-performing retailer is crucial for optimizing distribution strategies, negotiating better terms, and focusing marketing efforts where they yield the most impact. This insight suggests that Reliance Fresh is a key partner and understanding the factors contributing to their high sales can inform strategies for other retailers.

**Source:** Sales Analytics Database


In [41]:

# response = agent.run(

# """
# What ingredients are present in Pringles Original?
# """

# )

# print(response.content)

In [32]:

response = agent.run(

"""
Which SKU generated the highest sales
and what allergens does it contain?
"""

)

print(response.content)

Generated SQL
SELECT SKU_NAME FROM sales GROUP BY SKU_NAME ORDER BY SUM(SALES_VALUE) DESC LIMIT 1



INFO Found 5 documents

**SKU with Highest Sales:**

*   Chocos

**Allergens:**

I couldn't find information about the allergens in "Chocos" in my knowledge base.

**Business Insight:**
"Chocos" generated the highest sales, indicating its strong market performance. However, without allergen information, it's difficult to provide comprehensive product details to consumers, which could be a missed opportunity for informed purchasing decisions. It's crucial to have readily available allergen information for all top-selling SKUs to ensure consumer safety and transparency.

**Source:**
*   Sales Analytics Database
*   Knowledge Base (no information found for "Chocos" allergens)


In [33]:

response = agent.run(

"""
Explain Promotion ROI and identify
the retailer with the highest promotion sales.
"""

)

print(response.content)

Generated SQL
SELECT RETAILER FROM sales WHERE PROMOTION = 'True' GROUP BY RETAILER ORDER BY SUM(SALES_VALUE) DESC LIMIT 1



INFO Found 5 documents

**Promotion ROI (Return on Investment)**

Promotion ROI is a key metric used to evaluate the effectiveness and profitability of a promotional campaign. It is calculated as:

**Promotion ROI = Incremental Margin / Promotion Spend**

In simpler terms, it measures the incremental profit generated by a promotion relative to the cost of running that promotion. A higher ROI indicates a more successful and profitable promotion.

Promotion success should also be evaluated using incremental volume and incremental margin, considering factors like incremental sales, promotion lift, and potential cannibalization of other products.

**Retailer with the Highest Promotion Sales**

Based on the available sales data, no records were found to identify the retailer with the highest promotion sales.

**Sources:**
*   Commercial FAQs (linked to Commercial Knowledge)
*   Commercial Policy (linked to Commercial Knowledge)
*   Promotion SOP (linked to Commercial Knowledge)


In [34]:
response = agent.run(

"""
Show retailers that require manual review
according to retailer matching guidelines.
"""

)

print(response.content)

INFO Found 5 documents

Based on the "Retailer Matching Guidelines" and "Retailer Matching Framework" documents from the Commercial Knowledge Base:

### Answer
Retailers that have a confidence score between **80% and 95%** during the matching process require a manual review. Specifically, the "Retailer Matching Guide" states that a confidence score between 80-95% requires manual review, while the "Retailer Matching Framework" specifies confidence between 85% and 95% for manual review.

### Business Insight
Manual review is crucial for ensuring accuracy in retailer matching when the automated system's confidence is not high enough for an automatic match, but not low enough for a rejection. This helps in maintaining data integrity across commercial systems by carefully verifying retailer records that have moderate similarity signals.

### Source
*   Retailer_Matching_Guide (Page: 1, Linked to: Commercial Knowledge)
*   Retailer_Matching_Framework (Page: 1, Linked to: Commercial Knowledge)


In [ ]:

response = agent.run(

"""
Explain Weighted Distribution.
"""

)

print(response.content)

In [39]:

print("="*80)
print("Commercial Intelligence Copilot")
print("="*80)

while True:

    question = input("\nAsk : ")

    if question.lower() in ["exit","quit"]:

        break

    response = agent.run(question)

    print("\n")

    print(response.content)

In [40]:

print("="*80)
print("Commercial Intelligence Copilot")
print("="*80)

while True:

    question = input("\nAsk : ")

    if question.lower() in ["exit","quit"]:

        break

    response = agent.run(question)

    print("\n")

    print(response.content)
    